In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
library(data.table)
library(digest)

In [2]:
args <- list(
    in_vcf="data/conditional/rare/combined/ukb_eur_wes_200k_chr21_maf0to5e-2_pLoF_damaging_missense.vcf.bgz",
    out_prefix="data/conditional/rare/combined/ukb_eur_wes_200k_chr21_maf0to5e-2_pLoF_damaging_missense_ld_BC_combined_primary_care",
    phenotype="BC_combined_primary_care",
    pheno_file="data/phenotypes/curated_covar_phenotypes_binary_200k.tsv",
    covariates="data/phenotypes/covars1.csv"
)


In [3]:
gsub_to_dosage <- function(gt){
    gt <- gsub("\\|", "\\\\", gt)
    gt <- gsub("0/1", "1", gt)
    gt <- gsub("1/0", "1", gt)
    gt <- gsub("0/0", "0", gt)
    gt <- gsub("1/1", "2", gt)
    return(gt)
}

hash_genotypes <- function(G, algo = "xxhash32"){
    require(digest)
    gt_string <- unlist(apply(G, 1, function(x) as.character(paste(x, collapse = '-'))))
    dosage_string <- gsub_to_dosage(gt_string)
    the_hash <- unlist(lapply(dosage_string, function(x) digest(x, algo=algo)))
    return(the_hash)
}

In [4]:
stopifnot(file.exists(args$in_vcf))
stopifnot(file.exists(args$pheno_file))
stopifnot(file.exists(args$covariates))
stopifnot(dir.exists(dirname(args$out_prefix)))

In [29]:
lines <- 7089
lines_per_chunk <- 1000
chunks <- ceiling(lines / lines_per_chunk)

[1] 8

In [31]:
header_cmd <- cmd <- paste0("zcat ", args$in_vcf, " | grep '#CHR' ")
header <- fread(cmd = header_cmd, header = FALSE)
header <- as.character(t(header)[,1])

[1] "#CHROM"  "POS"     "ID"      "REF"     "ALT"     "QUAL"    "FILTER" 
    [8] "INFO"    "FORMAT"  "1705888" "3544034" "1065590" "1130642" "1237482"
   [15] "2351928" "3310264" "5800421" "3708144" "1728817" "1663370" "1116994"
   [22] "5254480" "2469141" "3099220" "1944907" "1759153" "2654239" "4290321"
   [29] "5746106" "5622376" "4282759" "2459760" "5223469" "4015841" "1233880"
   [36] "1056021" "2251532" "3999334" "4791741" "2839607" "5201370" "2891431"
   [43] "2428518" "2566462" "1204799" "2556278" "5123554" "2008606" "5465675"
   [50] "3997469" "3161144" "1547561" "2480913" "3313152" "1185247" "5204183"
   [57] "2731723" "4749899" "1314648" "1795433" "1942376" "4110487" "5767264"
   [64] "3999627" "3214056" "3556035" "1418092" "4785067" "2759451" "4849039"
   [71] "4046434" "5867275" "2862267" "2603624" "3398040" "4118210" "5944693"
   [78] "1337856" "3019362" "5295624" "2630554" "5029846" "4906545" "1970243"
   [85] "2237845" "5043109" "4624166" "1824199" "5125842" "4659790" "1911540"
   [92] "4670967" "3967443" "3166988" "1371232" "1301095" "5459352" "1532409"
   [99] "1661364" "4375686" "4880350" "1208529" "3854622" "4311257" "3453817"
  [106] "3879171" "5602611" "2724793" "5114706" "2962037" "4930708" "3511932"
  [113] "5432264" "4626529" "3274212" "6017705" "3235064" "2667735" "4125528"
  [120] "4263821" "2380441" "4750642" "2891054" "3824851" "3670917" "2067341"
  [127] "3053001" "5708521" "1340372" "1666963" "5546257" "1912963" "4725959"
  [134] "2450143" "1006685" "4803161" "3609162" "4555669" "4497348" "4281245"
  [141] "5041328" "4396795" "5555593" "4159474" "4265939" "1298172" "3627163"
  [148] "3580010" "5366456" "4049423" "1430841" "2771666" "4558728" "5084174"
  [155] "4610447" "4985299" "4057246" "3547740" "3727934" "3828066" "1544351"
  [162] "3738340" "2180415" "4953165" "2225545" "3962383" "1234576" "2144121"
  [169] "3896268" "4216690" "2441099" "3795825" "1559743" "3194107" "3820998"
  [176] "2941159" "2463648" "5021502" "4628813" "4056692" "3317068" "4779253"
  [183] "4874169" "3849378" "2129083" "4589876" "4041291" "1458307" "4605300"
  [190] "2724157" "1131087" "3096118" "1110070" "4293464" "1602765" "1197384"
  [197] "1327244" "3465033" "1393091" "2910267" "1290889" "1021041" "5950288"
  [204] "2324481" "4534519" "3420695" "1417504" "2562318" "4908345" "4190707"
  [211] "1195214" "2984992" "4775473" "2098099" "2472940" "2452657" "1679456"
  [218] "2038893" "2950645" "3834757" "3243051" "3266420" "4840030" "3227951"
  [225] "4528900" "2641063" "2879817" "4900656" "4341176" "2455313" "5677299"
  [232] "1063365" "1360779" "3765737" "3285932" "3020679" "4695609" "5441587"
  [239] "3475054" "5357848" "4491712" "2114726" "5066484" "3017459" "5109269"
  [246] "3344471" "5390647" "1483458" "4062737" "2335843" "6017640" "3793747"
  [253] "5587095" "2828558" "5028033" "3426959" "4136501" "3904739" "2645247"
  [260] "4633627" "5358812" "5036815" "3578426" "1623753" "3425087" "4629250"
  [267] "3105349" "1462714" "2507249" "3909385" "5978987" "1231386" "4826446"
  [274] "4415015" "2178766" "3354431" "1176524" "2073460" "4547249" "2874206"
  [281] "3157507" "2185708" "3092496" "2167796" "3636902" "4057316" "2506774"
  [288] "4139905" "5080226" "4914850" "4886583" "3670358" "1020702" "5961853"
  [295] "1460192" "4498121" "5309471" "2690552" "5163060" "2092402" "2662896"
  [302] "4403121" "2829488" "3225181" "1016284" "2913800" "3362540" "3298771"
  [309] "1417681" "3684769" "2914313" "3785957" "2089723" "5450712" "3887195"
  [316] "1900610" "1469717" "4050453" "3885014" "1205179" "2648307" "1090536"
  [323] "2608866" "1768082" "5402693" "5289659" "4334295" "2408378" "4050632"
  [330] "5091947" "3886741" "1568289" "3402117" "2848891" "1143552" "3500469"
  [337] "4934305" "5391936" "4306527" "1618686" "4883920" "1996819" "3762792"
  [344] "5593422" "5005971" "4653415" "3232692" "5951448" "2087555" "5996400"
  [351] "4805846" "2699563" "5638110" "3347561" "3839860" "2530045" "4430346"
  [358] "1887663" "1944666

In [43]:
vcf_lines

ERROR: Error in eval(expr, envir, enclos): object 'vcf_lines' not found


In [41]:
idx_start <- 0
idx_end <- lines_per_chunk
for (chunk in 1:chunks){
    
    msg <- paste("reading chunk",chunk,"of",chunks)
    write(msg, stdout())
    idx_end <- idx_end + lines_per_chunk
    idx_start <- idx_start + lines_per_chunk
    print(paste(idx_start,'-',idx_end))
    
}

reading chunk 1 of 8
[1] "0 - 1000"
reading chunk 2 of 8
[1] "1000 - 2000"
reading chunk 3 of 8
[1] "2000 - 3000"
reading chunk 4 of 8
[1] "3000 - 4000"
reading chunk 5 of 8
[1] "4000 - 5000"
reading chunk 6 of 8
[1] "5000 - 6000"
reading chunk 7 of 8
[1] "6000 - 7000"
reading chunk 8 of 8
[1] "7000 - 8000"


In [18]:
#md sed -n 5,8p


#iter_ <- 0
#stop_line <- 

cmd <- paste0("zcat ", args$in_vcf, " | grep -v '##' | sed -n 5,8p" )
cmd

[1] "zcat data/conditional/rare/combined/ukb_eur_wes_200k_chr21_maf0to5e-2_pLoF_damaging_missense.vcf.bgz | grep -v '##' | sed -n 5,8p"

In [19]:


# read in VCF
#cmd <- paste0("zcat ", args$in_vcf, " | grep -v '##' | head -n 500" )
d <- fread(cmd = cmd)

In [42]:
head(d)

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V176915,V176916,V176917,V176918,V176919,V176920,V176921,V176922,V176923,V176924
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr21,7,ENSG00000142173,fMTb,okHJ,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr21,9,ENSG00000142185,nibn,ppVn,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr21,11,ENSG00000142192,uFwA,aBgc,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr21,12,ENSG00000142197,TACC,dXNt,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [5]:
path <- "data/conditional/rare/combined/test.txt.gz"
d <- fread(path, showProgress = TRUE, verbose = TRUE)

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [6]:
dim(d)

[1]   7088 176924

In [63]:
# get columns with genotypes / metaid
genotype_cols <- suppressWarnings(!is.na(as.numeric(colnames(d))))
id_cols <- suppressWarnings(is.na(as.numeric(colnames(d))))

In [73]:
# Need to ensure that G is all numerics
G <- d[,genotype_cols, with = FALSE]
suppressWarnings(G[, names(G) := lapply(.SD, as.numeric)])

In [65]:
# read in phenotypes
phenotype <- args$phenotype
pheno_df <- fread(args$pheno_file)
stopifnot(phenotype %in% colnames(pheno_df))

# are there samples with missing covariates?
col_cov <- unlist(strsplit(readLines(args$covariates), split = ","))
lst <- lapply(col_cov, function(col){row_ok <- is.na(pheno_df[[col]])})
missing_cov <- rowSums(do.call(cbind, lst)) > 0
pheno_df <- pheno_df[!missing_cov, ]
msg <- paste("Note: Removed", sum(missing_cov),"samples with missing covariates.")
write(msg, stdout())

# get defined phenotypes
defined_phenos <- !is.na(pheno_df[[phenotype]])
eid_with_defined_phenos <- pheno_df$eid[defined_phenos]
stopifnot(length(eid_with_defined_phenos) > 0)

Note: Removed 20 samples with missing covariates.


In [66]:
# get subset of G which contains defiend phenotypes
G_with_defined_phenos <- colnames(G) %in% eid_with_defined_phenos
G_subset <- G[,G_with_defined_phenos, with = FALSE]

In [67]:
# Get allele count and hash for genotypes
G_subset_AC <- rowSums(G_subset, na.rm = TRUE)
G_subset_hash <- hash_genotypes(G_subset, "xxhash32")

In [69]:
id <- d[,id_cols, with = FALSE]
out <- data.table(
    chr = id$`#CHROM`, 
    pos = id$POS, 
    id = id$ID, 
    ref = id$REF,
    alt = id$ALT
)

In [70]:
# generate allele count outfile
outfile_ac <- paste0(args$out_prefix,"_AC.txt.gz")
out[[phenotype]] <- G_subset_AC
#fwrite(out, outfile_ac, sep = '\t')

In [71]:
# generate allele count outfile
outfile_hash <- paste0(args$out_prefix,"_AC.txt.gz")
out[[phenotype]] <- G_subset_hash
#fwrite(out, outfile_hash, sep = '\t')

In [72]:
out

chr,pos,id,ref,alt,BC_combined_primary_care
<chr>,<int>,<chr>,<chr>,<chr>,<chr>
chr21,1,ENSG00000141956,UpSX,zBVE,9080e1ea
chr21,2,ENSG00000141959,mGzq,EoIP,5a4352bc
chr21,4,ENSG00000142156,UpCp,rWjl,01ed5d56
chr21,7,ENSG00000142173,fMTb,okHJ,ae8fbd0d
chr21,9,ENSG00000142185,nibn,ppVn,67cba766
chr21,11,ENSG00000142192,uFwA,aBgc,a3dddb76
chr21,12,ENSG00000142197,TACC,dXNt,9b543033
chr21,13,ENSG00000142207,tdbv,TvnG,42f34ac5
chr21,28,ENSG00000155313,fVIj,TKUz,cb0e8b39
